In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Insert warehouse name here
warehouse_name = "RT"
base = "Lucas_Systems_Capstone_Project"

tables = {
    f"{warehouse_name}_Activity": f"../data/database_backups_csv/{warehouse_name}/{warehouse_name}_Activity.csv",
    f"{warehouse_name}_Locations": f"../data/database_backups_csv/{warehouse_name}/{warehouse_name}_Locations.csv",
    f"{warehouse_name}_Products": f"../data/database_backups_csv/{warehouse_name}/{warehouse_name}_Products.csv",
}

column_names = {
    f"{warehouse_name}_Activity": ["ActivityCode","UserID","WorkCode","AssignmentID","ProductID","Quantity","Timestamp","LocationID"],
    f"{warehouse_name}_Locations": ["LocationID","Aisle","Bay","Level","Slot"],
    f"{warehouse_name}_Products": ["ProductID","ProductCode","UnitOfMeasure","Weight","Cube","Length","Width","Height"],
}
#data/database_backups_csv/RT/RT_Activity.csv
dfs = {}

# Run this if there are no column names
for name, fp in tables.items():
    dfs[name] = pd.read_csv(fp, header=None, names=column_names[name])

# Load distance matrix
path = f"../data/distance_matrices/distance_matrix_{warehouse_name}.csv"
Distance = pd.read_csv(path, index_col=0)

for c in Distance.columns:
    Distance[c] = pd.to_numeric(Distance[c], errors="coerce")


In [2]:
for t in [f"{warehouse_name}_Activity", 
          f"{warehouse_name}_Locations", 
          f"{warehouse_name}_Products"]:

    print("=" * 80)
    print(f"Table: {t}")

    df = dfs[t]
    print(f"Dimensions: ({df.shape[0]} rows, {df.shape[1]} columns)\n")

    display(df.head(3))

    schema_df = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "n_unique": df.nunique(dropna=True),
    })

    num_df = df.select_dtypes(include="number")
    schema_df["min"] = num_df.min()
    schema_df["max"] = num_df.max()
    schema_df["mean"] = num_df.mean()

    display(schema_df)
    print("\n")


Table: RT_Activity
Dimensions: (225358 rows, 8 columns)



,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID
0,AssignmentOpen,534,2.0,4477546,NaN,NaN,2024-05-28 06:42:22.580,NaN
1,AssignmentOpen,534,2.0,4477547,NaN,NaN,2024-05-28 06:42:23.587,NaN
2,AssignmentOpen,534,2.0,4477548,NaN,NaN,2024-05-28 06:42:24.580,NaN


,dtype,n_missing,n_unique,min,max,mean
ActivityCode,object,0,2,NaN,NaN,NaN
UserID,int64,0,71,77.0,596.0,4.758759e+02
WorkCode,float64,1014,11,1.0,15.0,6.814726e+00
AssignmentID,int64,0,37124,309447.0,4514752.0,4.413726e+06
ProductID,float64,131183,6623,11488.0,28183.0,1.731315e+04
Quantity,float64,131183,184,1.0,1589.0,4.814579e+00
Timestamp,object,0,218345,NaN,NaN,NaN
LocationID,float64,131183,7822,96111.0,3316681.0,2.102896e+05




Table: RT_Locations
Dimensions: (20411 rows, 5 columns)



,LocationID,Aisle,Bay,Level,Slot
0,96111,211,102,10,NaN
1,96112,211,122,10,NaN
2,96113,211,122,30,NaN


,dtype,n_missing,n_unique,min,max,mean
LocationID,int64,0,20411,96111.0,3316681.0,257227.227818
Aisle,object,0,74,NaN,NaN,NaN
Bay,int64,0,884,1.0,962.0,396.863652
Level,object,0,75,NaN,NaN,NaN
Slot,float64,20411,0,NaN,NaN,NaN




Table: RT_Products
Dimensions: (16696 rows, 8 columns)



,ProductID,ProductCode,UnitOfMeasure,Weight,Cube,Length,Width,Height
0,11488,10027,NaN,22.00,NaN,23.0,12.75,15.0
1,11489,10068,NaN,13.68,NaN,15.5,5.50,12.5
2,11490,10076,NaN,21.75,NaN,23.0,12.75,15.0


,dtype,n_missing,n_unique,min,max,mean
ProductID,int64,0,16696,11488.00,28183.0,19835.500000
ProductCode,object,0,16696,NaN,NaN,NaN
UnitOfMeasure,float64,16696,0,NaN,NaN,NaN
Weight,float64,8172,3404,0.00,4240.0,235.919059
Cube,float64,15475,1,0.00,0.0,0.000000
Length,float64,8707,343,1.00,50.5,20.689750
Width,float64,8707,250,1.00,42.0,15.113250
Height,float64,8707,279,0.25,45.0,12.503017


In [3]:
display(Distance.head())

dist_long = (
    Distance.stack(dropna=False)
    .rename("distance")
    .reset_index()
    .rename(columns={"level_0": "FromLoc", "level_1": "ToLoc"})
)

display(dist_long.head())

,403:116,403:120,403:130,403:134,403:144,403:148,403:158,403:162,403:172,403:176,...,725End,750End,750Start,725Start,800Start,850Start,825Start,800End,850End,200End
403:116,0,27,54,81,108,135,162,189,216,243,...,4121,4020,4716,4817,4719,4623,4815,4047,3951,2428
403:120,27,0,27,54,81,108,135,162,189,216,...,4094,3993,4689,4790,4692,4596,4788,4020,3924,2401
403:130,54,27,0,27,54,81,108,135,162,189,...,4067,3966,4662,4763,4665,4569,4761,3993,3897,2374
403:134,81,54,27,0,27,54,81,108,135,162,...,4040,3939,4635,4736,4638,4542,4734,3966,3870,2347
403:144,108,81,54,27,0,27,54,81,108,135,...,4013,3912,4608,4709,4611,4515,4707,3939,3843,2320


/var/folders/hs/r4ck14j54v17d80mrc1wt5kw0000gn/T/ipykernel_10614/2095669890.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  Distance.stack(dropna=False)


,FromLoc,ToLoc,distance
0,403:116,403:116,0
1,403:116,403:120,27
2,403:116,403:130,54
3,403:116,403:134,81
4,403:116,403:144,108


In [4]:
activity_key = f"{warehouse_name}_Activity"
locations_key = f"{warehouse_name}_Locations"
products_key = f"{warehouse_name}_Products"

# Activity
dfs[activity_key]["ProductID"] = pd.to_numeric(dfs[activity_key]["ProductID"], errors="coerce").astype("Int64")
dfs[activity_key]["Quantity"]  = pd.to_numeric(dfs[activity_key]["Quantity"], errors="coerce").astype("Int64")
dfs[activity_key]["LocationID"] = pd.to_numeric(dfs[activity_key]["LocationID"], errors="coerce").astype("Int64")
dfs[activity_key]["Timestamp"] = pd.to_datetime(dfs[activity_key]["Timestamp"], errors="coerce")
dfs[activity_key]["UserID"] = dfs[activity_key]["UserID"].astype(str)
dfs[activity_key]["WorkCode"] = dfs[activity_key]["WorkCode"].astype(str)
dfs[activity_key]["AssignmentID"] = dfs[activity_key]["AssignmentID"].astype(str)

dfs[activity_key] = dfs[activity_key].dropna(subset=["Timestamp"]).copy()

# Locations
dfs[locations_key]["LocationID"] = pd.to_numeric(dfs[locations_key]["LocationID"], errors="coerce").astype("Int64")
for col in ["Bay", "Level", "Slot"]:
    dfs[locations_key][col] = pd.to_numeric(dfs[locations_key][col], errors="coerce").astype("Int64")

# Products
dfs[products_key]["ProductID"] = pd.to_numeric(dfs[products_key]["ProductID"], errors="coerce").astype("Int64")
dfs[products_key] = dfs[products_key][["ProductID", "ProductCode", "UnitOfMeasure", "Weight", "Cube"]]

Activity = dfs[activity_key]
Locations = dfs[locations_key]
Products = dfs[products_key]


In [5]:
df_work = Activity.copy()
df_work = df_work.sort_values(["UserID", "Timestamp"]).reset_index(drop=True)

g = df_work.groupby("UserID", sort=False)
df_work["Prev_Timestamp"] = g["Timestamp"].shift(1)
df_work["Prev_LocationID"] = g["LocationID"].shift(1)

df_work["Time_Delta_sec"] = (
    df_work["Timestamp"] - df_work["Prev_Timestamp"]
).dt.total_seconds()

df_work.loc[df_work["Time_Delta_sec"] > 180 * 60, "Time_Delta_sec"] = np.nan

Activity_prepped = df_work

In [6]:
display(Activity_prepped.head())

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,Time_Delta_sec
0,AssignmentOpen,102,5.0,4477558,<NA>,<NA>,2024-05-28 07:49:08.177,<NA>,NaT,<NA>,NaN
1,PickPut,102,5.0,4477558,15441,3,2024-05-28 07:49:21.627,112687,2024-05-28 07:49:08.177,<NA>,13.450
2,PickPut,102,5.0,4477558,14469,6,2024-05-28 07:54:14.920,104027,2024-05-28 07:49:21.627,112687,293.293
3,PickPut,102,5.0,4477558,11772,6,2024-05-28 07:55:19.720,96167,2024-05-28 07:54:14.920,104027,64.800
4,PickPut,102,5.0,4477558,11532,11,2024-05-28 07:59:02.700,115278,2024-05-28 07:55:19.720,96167,222.980


In [7]:
df_joined = Activity_prepped.merge(Products, on="ProductID", how="left")
df_joined = df_joined.merge(Locations, on="LocationID", how="left")

df_joined = df_joined.merge(
    Locations[["LocationID","Aisle","Bay","Level","Slot"]].rename(columns={
        "LocationID": "Prev_LocationID",
        "Aisle": "Prev_Aisle",
        "Bay": "Prev_Bay",
        "Level": "Prev_Level",
        "Slot": "Prev_Slot",
    }),
    on="Prev_LocationID",
    how="left"
)

display(df_joined.head())

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Weight,Cube,Aisle,Bay,Level,Slot,Prev_Aisle,Prev_Bay,Prev_Level,Prev_Slot
0,AssignmentOpen,102,5.0,4477558,<NA>,<NA>,2024-05-28 07:49:08.177,<NA>,NaT,<NA>,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>
1,PickPut,102,5.0,4477558,15441,3,2024-05-28 07:49:21.627,112687,2024-05-28 07:49:08.177,<NA>,...,3304.0,NaN,222,142,10,<NA>,NaN,<NA>,<NA>,<NA>
2,PickPut,102,5.0,4477558,14469,6,2024-05-28 07:54:14.920,104027,2024-05-28 07:49:21.627,112687,...,3192.0,NaN,231,146,10,<NA>,222,142,10,<NA>
3,PickPut,102,5.0,4477558,11772,6,2024-05-28 07:55:19.720,96167,2024-05-28 07:54:14.920,104027,...,3612.0,NaN,231,114,20,<NA>,231,146,10,<NA>
4,PickPut,102,5.0,4477558,11532,11,2024-05-28 07:59:02.700,115278,2024-05-28 07:55:19.720,96167,...,3566.0,NaN,232,110,10,<NA>,231,114,20,<NA>


In [8]:
# modify the code above as the format for this warehouse is different from OE, it is a : in between Aisle and Bay, so we need to split it first before converting to numeric
df_detailed = df_joined.copy()

df_detailed["Aisle2"] = pd.to_numeric(df_detailed["Aisle"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
df_detailed["Bay2"] = pd.to_numeric(df_detailed["Bay"], errors="coerce").astype("Int64").astype(str).str.zfill(2)

df_detailed["Prev_Aisle2"] = pd.to_numeric(df_detailed["Prev_Aisle"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
df_detailed["Prev_Bay2"] = pd.to_numeric(df_detailed["Prev_Bay"], errors="coerce").astype("Int64").astype(str).str.zfill(2)

df_detailed["LocKey"] = df_detailed["Aisle2"] + ":" + df_detailed["Bay2"]
df_detailed["PrevLocKey"] = df_detailed["Prev_Aisle2"] + ":" + df_detailed["Prev_Bay2"]

df_detailed = df_detailed.merge(
    dist_long,
    left_on=["LocKey", "PrevLocKey"],
    right_on=["FromLoc", "ToLoc"],
    how="left"
).rename(columns={"distance": "Travel_Distance"}).drop(columns=["FromLoc", "ToLoc"])

display(df_detailed.head())


,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Prev_Bay,Prev_Level,Prev_Slot,Aisle2,Bay2,Prev_Aisle2,Prev_Bay2,LocKey,PrevLocKey,Travel_Distance
0,AssignmentOpen,102,5.0,4477558,<NA>,<NA>,2024-05-28 07:49:08.177,<NA>,NaT,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>:<NA>,<NA>:<NA>,NaN
1,PickPut,102,5.0,4477558,15441,3,2024-05-28 07:49:21.627,112687,2024-05-28 07:49:08.177,<NA>,...,<NA>,<NA>,<NA>,222,142,<NA>,<NA>,222:142,<NA>:<NA>,NaN
2,PickPut,102,5.0,4477558,14469,6,2024-05-28 07:54:14.920,104027,2024-05-28 07:49:21.627,112687,...,142,10,<NA>,231,146,222,142,231:146,222:142,1208.0
3,PickPut,102,5.0,4477558,11772,6,2024-05-28 07:55:19.720,96167,2024-05-28 07:54:14.920,104027,...,146,10,<NA>,231,114,231,146,231:114,231:146,192.0
4,PickPut,102,5.0,4477558,11532,11,2024-05-28 07:59:02.700,115278,2024-05-28 07:55:19.720,96167,...,114,20,<NA>,232,110,231,114,232:110,231:114,340.0


In [19]:
# diaplay first 50 rows of the final cleaned dataset with userid = 102
display(df_detailed[df_detailed["UserID"] == "102"].head())

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Prev_Bay,Prev_Level,Prev_Slot,Aisle2,Bay2,Prev_Aisle2,Prev_Bay2,LocKey,PrevLocKey,Travel_Distance
0,AssignmentOpen,102,5.0,4477558,<NA>,<NA>,2024-05-28 07:49:08.177,<NA>,NaT,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>:<NA>,<NA>:<NA>,NaN
1,PickPut,102,5.0,4477558,15441,3,2024-05-28 07:49:21.627,112687,2024-05-28 07:49:08.177,<NA>,...,<NA>,<NA>,<NA>,222,142,<NA>,<NA>,222:142,<NA>:<NA>,NaN
2,PickPut,102,5.0,4477558,14469,6,2024-05-28 07:54:14.920,104027,2024-05-28 07:49:21.627,112687,...,142,10,<NA>,231,146,222,142,231:146,222:142,1208.0
3,PickPut,102,5.0,4477558,11772,6,2024-05-28 07:55:19.720,96167,2024-05-28 07:54:14.920,104027,...,146,10,<NA>,231,114,231,146,231:114,231:146,192.0
4,PickPut,102,5.0,4477558,11532,11,2024-05-28 07:59:02.700,115278,2024-05-28 07:55:19.720,96167,...,114,20,<NA>,232,110,231,114,232:110,231:114,340.0


In [16]:
# Percentage of ActivityCode == AssignmentOpen
assignment_open_pct = (df_detailed["ActivityCode"] == "AssignmentOpen").mean()
print(f"Percentage of ActivityCode == AssignmentOpen: {assignment_open_pct:.2%}")

Percentage of ActivityCode == AssignmentOpen: 58.21%


In [10]:
# Identify indices of 'AssignmentOpen'
open_indices = df_detailed[df_detailed["ActivityCode"] == "AssignmentOpen"].index

# Identify indices of the first activity immediately following an 'AssignmentOpen'
first_activity_indices = open_indices + 1

# Combine them into one set of indices to remove
to_drop = open_indices.union(first_activity_indices).intersection(df_detailed.index)

# Create the final cleaned dataset
Detailed_Data = df_detailed.drop(to_drop).reset_index(drop=True)

In [20]:
# diaplay first 50 rows of the final cleaned dataset with userid = 102
display(df_detailed[df_detailed["UserID"] == "102"].head(5))

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Prev_Bay,Prev_Level,Prev_Slot,Aisle2,Bay2,Prev_Aisle2,Prev_Bay2,LocKey,PrevLocKey,Travel_Distance
0,AssignmentOpen,102,5.0,4477558,<NA>,<NA>,2024-05-28 07:49:08.177,<NA>,NaT,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>:<NA>,<NA>:<NA>,NaN
1,PickPut,102,5.0,4477558,15441,3,2024-05-28 07:49:21.627,112687,2024-05-28 07:49:08.177,<NA>,...,<NA>,<NA>,<NA>,222,142,<NA>,<NA>,222:142,<NA>:<NA>,NaN
2,PickPut,102,5.0,4477558,14469,6,2024-05-28 07:54:14.920,104027,2024-05-28 07:49:21.627,112687,...,142,10,<NA>,231,146,222,142,231:146,222:142,1208.0
3,PickPut,102,5.0,4477558,11772,6,2024-05-28 07:55:19.720,96167,2024-05-28 07:54:14.920,104027,...,146,10,<NA>,231,114,231,146,231:114,231:146,192.0
4,PickPut,102,5.0,4477558,11532,11,2024-05-28 07:59:02.700,115278,2024-05-28 07:55:19.720,96167,...,114,20,<NA>,232,110,231,114,232:110,231:114,340.0


In [12]:
# Display column names and data types of the final cleaned dataset
display(Detailed_Data.dtypes)

ActivityCode               object
UserID                     object
WorkCode                   object
AssignmentID               object
ProductID                   Int64
Quantity                    Int64
Timestamp          datetime64[ns]
LocationID                  Int64
Prev_Timestamp     datetime64[ns]
Prev_LocationID             Int64
Time_Delta_sec            float64
ProductCode                object
UnitOfMeasure             float64
Weight                    float64
Cube                      float64
Aisle                      object
Bay                         Int64
Level                       Int64
Slot                        Int64
Prev_Aisle                 object
Prev_Bay                    Int64
Prev_Level                  Int64
Prev_Slot                   Int64
Aisle2                     object
Bay2                       object
Prev_Aisle2                object
Prev_Bay2                  object
LocKey                     object
PrevLocKey                 object
Travel_Distanc

In [13]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

Detailed_Data.to_parquet(output_dir / f"{warehouse_name.lower()}_detailed.parquet", index=False)
Activity_prepped.to_parquet(output_dir / f"{warehouse_name.lower()}_activity_prepped.parquet", index=False)
df_joined.to_parquet(output_dir / f"{warehouse_name.lower()}_joined.parquet", index=False)

print(f"Successfully exported all {warehouse_name} files to {output_dir}")


Successfully exported all RT files to ../data/processed


In [14]:
# Print number of rows in Activity vs in Detailed_Data, and percentage reduced
nrow_original = Activity.shape[0]
nrow_cleaned = Detailed_Data.shape[0]
retained_pc = nrow_cleaned / nrow_original * 100
print(nrow_original)
print(nrow_cleaned)
print(f"Percentage retained: {retained_pc}")

225358
82147
Percentage retained: 36.451778947275


# Extra

In [15]:
df = Activity_prepped.copy()
df = df.dropna(subset=["ProductID", "Time_Delta_sec"]).copy()

df["Prev_ProductID"] = df.groupby("UserID")["ProductID"].shift(1)
df_pairs = df[df["ProductID"] == df["Prev_ProductID"]].copy()

product_pick_times = (
    df_pairs.groupby("ProductID")
            .agg(
                n_pairs=("Time_Delta_sec", "size"),
                avg_pick_time_sec=("Time_Delta_sec", "mean"),
                median_pick_time_sec=("Time_Delta_sec", "median"),
                std_pick_time_sec=("Time_Delta_sec", "std")
            )
            .reset_index()
            .sort_values("ProductID")
)

display(product_pick_times.head())
product_pick_times.to_csv(f"../data/processed/product_pick_times.csv", index=False)

,ProductID,n_pairs,avg_pick_time_sec,median_pick_time_sec,std_pick_time_sec
0,11489,5,31.253200,33.0830,30.759316
1,11490,2,4.142000,4.1420,0.328098
2,11491,3,42.603333,16.7970,45.954093
3,11500,2,8.564500,8.5645,7.398458
4,11502,1,25.080000,25.0800,NaN
